# 23 — Lieb-Mattis States Inside the DFS

**Repo:** `decoherence-free-sensing`  
**Notebook role:** many-body state-design layer  
**Previous:** `17_common_mode_rejection.ipynb`  
**Next:** `29_quantum_fisher_scaling.ipynb`

Notebook 23 transitions from noise rejection to many-body state design.

The previous notebooks established the admissible constraint:

\[
J_z^+ = J_z^A + J_z^B
\]

should be fixed so common-mode phase noise becomes physically unobservable.

Notebook 23 asks the next question:

**Which entangled states live inside that admissible DFS while retaining differential-phase sensitivity?**

The central example is a Lieb-Mattis state:

\[
|\psi_{\rm LM}\rangle
=
\sum_x (-1)^x c_x |x\rangle_A |\bar{x}\rangle_B .
\]

It satisfies

\[
J_z^+|\psi_{\rm LM}\rangle=0,
\]

while generally retaining sensitivity through

\[
J_z^-|\psi_{\rm LM}\rangle \neq 0.
\]

## Engineering statement

Lieb-Mattis states are admissible entanglement because they live inside the DFS while retaining differential-phase sensitivity.


## Notebook outputs

Running this notebook creates:

- `results/figures/23_central_dfs_basis.png`
- `results/figures/23_lieb_mattis_amplitudes.png`
- `results/figures/23_state_lives_inside_dfs.png`
- `results/figures/23_pair_correlation_structure.png`
- `results/figures/23_lieb_mattis_design_rule.png`
- `results/lieb_mattis_summary.csv`
- `results/lieb_mattis_summary.json`
- `results/decoherence_free_sensing_lieb_mattis_outputs.zip`


In [ ]:
from pathlib import Path
import json
import math
import zipfile
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from matplotlib.patches import FancyBboxPatch, FancyArrowPatch

ROOT = Path.cwd()
RESULTS = ROOT / "results"
FIGURES = RESULTS / "figures"
SITE = ROOT / "site"

for path in [RESULTS, FIGURES, SITE]:
    path.mkdir(parents=True, exist_ok=True)

plt.rcParams.update({
    "figure.dpi": 160,
    "savefig.dpi": 220,
    "font.size": 12,
    "axes.titlesize": 16,
    "axes.labelsize": 12,
    "legend.fontsize": 11,
})


## 1. Central DFS basis

For a two-ensemble sensor, a central DFS basis pairs every excitation number in ensemble A with the complementary excitation number in ensemble B.

For a small symmetric example with \(n\) atoms in each ensemble,

\[
|x\rangle_A |\bar{x}\rangle_B
\]

has a fixed total \(J_z^+\) value.

That fixed sector is what makes common-mode phase noise global.


In [ ]:
def binomial(n, k):
    return math.comb(n, k)

n = 10
x_vals = np.arange(n + 1)
complement = n - x_vals
sector_m = x_vals + complement - n  # centered representation = 0 always

basis = pd.DataFrame({
    "x_A": x_vals,
    "xbar_B": complement,
    "Jz_plus_sector": sector_m,
    "degeneracy_proxy": [binomial(n, int(x)) for x in x_vals],
})

fig, ax = plt.subplots(figsize=(11.5, 5.8))

for x, xb in zip(x_vals, complement):
    ax.plot([x, x], [0.15, 0.75], linewidth=1.4, alpha=0.35)
    ax.scatter([x], [0.75], s=85)
    ax.scatter([x], [0.15], s=85)
    ax.text(x, 0.86, f"{x}", ha="center", va="bottom", fontsize=9)
    ax.text(x, 0.03, f"{xb}", ha="center", va="top", fontsize=9)

ax.text(-0.65, 0.75, "A excitations", ha="right", va="center", fontsize=12, fontweight="bold")
ax.text(-0.65, 0.15, "B complement", ha="right", va="center", fontsize=12, fontweight="bold")

ax.set_xlim(-1, n + 1)
ax.set_ylim(-0.12, 1.05)
ax.set_axis_off()
ax.set_title("Central DFS Basis: Pair Each A Configuration with Its Complement in B", pad=20)

ax.text(
    n/2,
    -0.08,
    r"Every paired basis element has fixed $J_z^+=0$.",
    ha="center",
    va="center",
    fontsize=14
)

path = FIGURES / "23_central_dfs_basis.png"
fig.savefig(path, bbox_inches="tight")
plt.show()

basis.head(), path


## 2. Lieb-Mattis amplitude construction

The Lieb-Mattis target state is a coherent superposition over the central DFS basis:

\[
|\psi_{\rm LM}\rangle
=
\sum_x (-1)^x c_x |x\rangle_A |\bar{x}\rangle_B .
\]

For a compact computational proxy, we use binomial-weighted amplitudes:

\[
c_x \propto \sqrt{\binom{n}{x}}.
\]

The alternating sign \((-1)^x\) encodes the singlet-like structure.


In [ ]:
weights = np.array([binomial(n, int(x)) for x in x_vals], dtype=float)
c = np.sqrt(weights)
c = c / np.linalg.norm(c)
signed_c = ((-1) ** x_vals) * c
prob = c**2

fig, ax = plt.subplots(figsize=(8.6, 5.8))
ax.axhline(0, linewidth=1.0)
ax.bar(x_vals, signed_c, alpha=0.85)
ax.set_title("Lieb-Mattis Amplitudes: Alternating Signed DFS Superposition")
ax.set_xlabel(r"central DFS basis index $x$")
ax.set_ylabel(r"signed amplitude $(-1)^x c_x$")
ax.grid(True, axis="y", alpha=0.25)

ax.text(
    0.5,
    0.86,
    r"$|\psi_{LM}\rangle=\sum_x (-1)^x c_x |x\rangle_A|\bar{x}\rangle_B$",
    transform=ax.transAxes,
    ha="center",
    va="center",
    fontsize=13,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.9)
)

path = FIGURES / "23_lieb_mattis_amplitudes.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 3. State support: inside versus outside the DFS

A valid decoherence-free entangled state should have support only inside one fixed \(J_z^+\) sector.

Here we compare:

- the Lieb-Mattis proxy state, supported only in the central sector,
- a leakage proxy state with small amplitude outside the central sector.

The engineering point is simple:

\[
J_z^+|\psi_{\rm LM}\rangle=0.
\]


In [ ]:
sectors = np.arange(-5, 6)
lm_support = np.zeros_like(sectors, dtype=float)
lm_support[sectors == 0] = 1.0

leaky_support = np.exp(-0.5 * (sectors / 1.7) ** 2)
leaky_support = leaky_support / leaky_support.sum()

fig, ax = plt.subplots(figsize=(8.6, 5.8))

ax.bar(sectors - 0.18, lm_support, width=0.36, alpha=0.85, label="Lieb-Mattis state")
ax.bar(sectors + 0.18, leaky_support, width=0.36, alpha=0.65, label="leaky superposition")
ax.set_title("Lieb-Mattis State Lives Inside One DFS Sector")
ax.set_xlabel(r"$J_z^+$ sector")
ax.set_ylabel("state support")
ax.grid(True, axis="y", alpha=0.25)
ax.legend()

ax.text(
    0,
    0.72,
    r"fixed sector: $J_z^+=0$",
    ha="center",
    va="center",
    fontsize=12,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.9)
)

path = FIGURES / "23_state_lives_inside_dfs.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 4. Pair-correlation structure

The Lieb-Mattis construction pairs each basis element in ensemble A with a complementary basis element in ensemble B.

This creates anti-correlated population structure:

\[
x_A + x_B = n.
\]

That anti-correlation is exactly what keeps the state in the central DFS sector.


In [ ]:
X, Y = np.meshgrid(x_vals, x_vals)
pair_weight = np.zeros_like(X, dtype=float)

for i, x in enumerate(x_vals):
    y = n - x
    pair_weight[y, x] = prob[i]

fig, ax = plt.subplots(figsize=(7.2, 6.2))
im = ax.imshow(pair_weight, origin="lower", aspect="auto")
ax.set_title("Pair-Correlation Structure: Complementary A/B Excitations")
ax.set_xlabel(r"A excitation index $x_A$")
ax.set_ylabel(r"B excitation index $x_B$")
ax.set_xticks(x_vals[::2])
ax.set_yticks(x_vals[::2])

cbar = fig.colorbar(im, ax=ax)
cbar.set_label("probability weight")

ax.text(
    0.5,
    0.94,
    r"support lies on $x_A+x_B=n$",
    transform=ax.transAxes,
    ha="center",
    va="center",
    fontsize=12,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.9)
)

path = FIGURES / "23_pair_correlation_structure.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 5. Differential sensitivity proxy

The same state that is protected from common-mode phase can still be sensitive to differential phase.

Inside the central DFS sector:

\[
J_z^+|\psi_{\rm LM}\rangle=0,
\]

but the differential generator has nonzero spread:

\[
\mathrm{Var}(J_z^-)>0.
\]

That variance is the resource that later connects to quantum Fisher information.


In [ ]:
# Proxy differential generator eigenvalues:
# If x_A = x and x_B = n-x, then a centered differential eigenvalue is proportional to x - (n-x) = 2x - n.
jz_minus = 2 * x_vals - n
mean_minus = np.sum(prob * jz_minus)
var_minus = np.sum(prob * (jz_minus - mean_minus) ** 2)

fig, ax = plt.subplots(figsize=(8.6, 5.8))
ax.bar(jz_minus, prob, alpha=0.85)
ax.axvline(mean_minus, linestyle="--", linewidth=1.7, label=fr"mean = {mean_minus:.2f}")
ax.set_title("Differential Generator Has Nonzero Spread Inside the DFS")
ax.set_xlabel(r"proxy $J_z^-$ eigenvalue")
ax.set_ylabel("probability weight")
ax.grid(True, axis="y", alpha=0.25)
ax.legend()

ax.text(
    0.5,
    0.84,
    fr"$\mathrm{{Var}}(J_z^-)\approx {var_minus:.2f}$",
    transform=ax.transAxes,
    ha="center",
    va="center",
    fontsize=13,
    bbox=dict(boxstyle="round,pad=0.3", facecolor="white", edgecolor="black", alpha=0.9)
)

# This figure is useful for the notebook, but the requested public set uses the five figures above.
# Save it as an auxiliary figure for later use.
aux_path = FIGURES / "23_differential_generator_spread_aux.png"
fig.savefig(aux_path, bbox_inches="tight")
plt.show()
aux_path


## 6. Lieb-Mattis design rule

The design rule is:

1. start inside the DFS,
2. pair complementary A/B configurations,
3. add alternating Lieb-Mattis signs,
4. preserve \(J_z^+\),
5. retain spread in \(J_z^-\).

That is the first admissible entangled object after common-mode rejection has specified the allowed subspace.


In [ ]:
fig, ax = plt.subplots(figsize=(13.0, 5.8))
ax.set_axis_off()
ax.set_xlim(0, 1)
ax.set_ylim(0, 1)

def draw_box(ax, xy, width, height, title, body=None, fontsize=15, facecolor="white", edgecolor="black", lw=1.8):
    x, y = xy
    box = FancyBboxPatch(
        (x, y), width, height,
        boxstyle="round,pad=0.025,rounding_size=0.04",
        linewidth=lw,
        edgecolor=edgecolor,
        facecolor=facecolor
    )
    ax.add_patch(box)
    ax.text(
        x + width/2,
        y + height*0.62 if body else y + height/2,
        title,
        ha="center",
        va="center",
        fontsize=fontsize,
        fontweight="bold"
    )
    if body:
        ax.text(
            x + width/2,
            y + height*0.28,
            body,
            ha="center",
            va="center",
            fontsize=11
        )

def draw_arrow(ax, p0, p1):
    ax.add_patch(FancyArrowPatch(
        p0, p1,
        arrowstyle="-|>",
        mutation_scale=18,
        linewidth=1.8
    ))

ax.text(0.5, 0.92, "Lieb-Mattis State Design Rule", ha="center", va="center", fontsize=23)

items = [
    (0.05, "Constrain", "central DFS\n$J_z^+=0$"),
    (0.285, "Pair", "complementary\nA/B basis"),
    (0.52, "Entangle", "alternating signs\n$(-1)^x c_x$"),
    (0.755, "Sense", "retain spread\nin $J_z^-$"),
]
w, h, y = 0.19, 0.32, 0.42

for x0, title, body in items:
    draw_box(ax, (x0, y), w, h, title, body)

for x0 in [0.245, 0.48, 0.715]:
    draw_arrow(ax, (x0, y + h/2), (x0 + 0.035, y + h/2))

ax.text(
    0.5,
    0.22,
    "Entanglement is admissible only after it satisfies the DFS constraint.",
    ha="center",
    va="center",
    fontsize=14
)

path = FIGURES / "23_lieb_mattis_design_rule.png"
fig.savefig(path, bbox_inches="tight")
plt.show()
path


## 7. Summary table


In [ ]:
summary = pd.DataFrame([
    {
        "item": "target_state",
        "value": "psi_LM = sum_x (-1)^x c_x |x>_A |xbar>_B",
        "engineering_role": "DFS-compatible entangled state"
    },
    {
        "item": "dfs_constraint",
        "value": "J_z^+ psi_LM = 0",
        "engineering_role": "common-mode phase becomes global"
    },
    {
        "item": "paired_basis",
        "value": "x_A + x_B = n",
        "engineering_role": "keeps support in the central DFS"
    },
    {
        "item": "alternating_sign",
        "value": "(-1)^x",
        "engineering_role": "singlet-like Lieb-Mattis structure"
    },
    {
        "item": "differential_generator",
        "value": "J_z^- psi_LM is not zero in general",
        "engineering_role": "preserves differential sensitivity"
    },
    {
        "item": "variance_proxy",
        "value": f"Var(J_z^-) approx {var_minus:.6f} for n={n} proxy",
        "engineering_role": "resource for phase sensitivity"
    },
    {
        "item": "design_rule",
        "value": "Constrain first; entangle inside the DFS second",
        "engineering_role": "state-design specification"
    },
])

summary_csv = RESULTS / "lieb_mattis_summary.csv"
summary_json = RESULTS / "lieb_mattis_summary.json"

summary.to_csv(summary_csv, index=False)
summary.to_json(summary_json, orient="records", indent=2)

summary


## 8. Export notebook outputs

This cell creates one zip archive containing all generated figures and summary files.


In [ ]:
zip_path = RESULTS / "decoherence_free_sensing_lieb_mattis_outputs.zip"

public_figures = [
    FIGURES / "23_central_dfs_basis.png",
    FIGURES / "23_lieb_mattis_amplitudes.png",
    FIGURES / "23_state_lives_inside_dfs.png",
    FIGURES / "23_pair_correlation_structure.png",
    FIGURES / "23_lieb_mattis_design_rule.png",
]

with zipfile.ZipFile(zip_path, "w", compression=zipfile.ZIP_DEFLATED) as zf:
    for file in public_figures:
        if file.exists():
            zf.write(file, file.relative_to(ROOT))
    aux = FIGURES / "23_differential_generator_spread_aux.png"
    if aux.exists():
        zf.write(aux, aux.relative_to(ROOT))
    zf.write(summary_csv, summary_csv.relative_to(ROOT))
    zf.write(summary_json, summary_json.relative_to(ROOT))

print(f"Created: {zip_path}")

# Optional Colab download:
try:
    from google.colab import files
    files.download(str(zip_path))
except Exception:
    pass

zip_path


## 9. Next notebook

Suggested next notebook:

`29_quantum_fisher_scaling.ipynb`

Core goal:

- connect differential-generator variance to quantum Fisher information,
- compare SQL, Heisenberg, and Lieb-Mattis QCRB references,
- show how DFS-compatible entanglement becomes metrological advantage.
